In [1]:
import pandas as pd
import html
import re

import os

In [2]:
INSTITUTION_ID = 0

In [ ]:
# import moma artworks csv
df = pd.read_json('../raw-museum-data/moma/Artworks.json')

In [31]:
# filter to only works that were made by this artist
# Assume the 'Artist' column contains a list of artist names for each artwork
artist_name = "Patty Chang"

In [32]:
artist_works = df[df['Artist'].apply(lambda artists: artist_name in artists if isinstance(artists, list) else False)]
artist_works.to_csv(f'./exports/{artist_name}_works.csv', index=False)
artist_works.head()

,Title,Artist,ConstituentID,ArtistBio,Nationality,BeginDate,EndDate,Gender,Date,Medium,...,ImageURL,OnView,Height (cm),Width (cm),Depth (cm),Weight (kg),Diameter (cm),Length (cm),Circumference (cm),Duration (sec.)
108264,In Love,[Patty Chang],[35426],"[American, born 1972]",[American],[1972],[0],[female],2001,"Two-channel video (color, sound)",...,https://www.moma.org/media/W1siZiIsIjIyNzc4NSJ...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,208.0
108573,Untitled (Eels),[Patty Chang],[35426],"[American, born 1972]",[American],[1972],[0],[female],1999,"Video (color, sound)",...,https://www.moma.org/media/W1siZiIsIjI1MTY2NSJ...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,960.0


In [33]:
artist_works = pd.read_csv(f'./exports/{artist_name}_works.csv')

In [34]:
# manually assign artist id
ARTIST_ID = 28

In [35]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return html.unescape(str(value)).strip()


def format_mdy(value):
    if pd.isna(value) or str(value).strip() == "":
        return ""
    
    s = str(value).split("T")[0].strip()
    dt = pd.to_datetime(s, errors="coerce")
    
    if pd.isna(dt):
        return s
    
    return f"{dt.month}/{dt.day}/{dt.year}"


def build_medium(row):
    parts = [
        clean_text(row.get("Medium")),
        clean_text(row.get("Classification")),
        clean_text(row.get("Department")),
    ]
    return " | ".join([p for p in parts if p])

In [36]:
converted = pd.DataFrame({
    "id": [""] * len(artist_works),
    "artist": [ARTIST_ID] * len(artist_works),
    "title": artist_works["Title"].apply(clean_text),
    "alt_text": [""] * len(artist_works),
    "description": [""] * len(artist_works),
    "date_created": artist_works["Date"].apply(clean_text),
    "date_acquired_or_updated": artist_works["DateAcquired"].apply(format_mdy),
    "institution": [INSTITUTION_ID] * len(artist_works),
    "medium": artist_works.apply(build_medium, axis=1),
    "image_url": artist_works["ImageURL"].apply(clean_text) if "ImageURL" in artist_works.columns else "",
})

output_csv = f"./formatted-exports/{artist_name}.csv"

if output_csv:
    if os.path.exists(output_csv):
        converted.to_csv(output_csv, mode="a", header=False, index=False)
    else:
        converted.to_csv(output_csv, index=False)

print(converted.to_csv(index=False))

id,artist,title,alt_text,description,date_created,date_acquired_or_updated,institution,medium,image_url
,28,In Love,,,2001,12/13/2011,0,"Two-channel video (color, sound) | Installation | Media and Performance",https://www.moma.org/media/W1siZiIsIjIyNzc4NSJdLFsicCIsImNvbnZlcnQiLCItcmVzaXplIDEwMjR4MTAyNFx1MDAzZSJdXQ.jpg?sha=216993a8a12e35ce
,28,Untitled (Eels),,,1999,12/13/2011,0,"Video (color, sound) | Video | Media and Performance",https://www.moma.org/media/W1siZiIsIjI1MTY2NSJdLFsicCIsImNvbnZlcnQiLCItcmVzaXplIDEwMjR4MTAyNFx1MDAzZSJdXQ.jpg?sha=082ddf40ceffdb2d

